In [ ]:
import os
os.environ['http_proxy'] = 'http://127.0.0.1:7897'
os.environ['https_proxy'] = 'http://127.0.0.1:7897'

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd

data = pd.read_csv('ChnSentiCorp_htl_all.csv')
# len(data)
data = data.dropna()

In [ ]:
from torch.utils.data import Dataset, DataLoader

class Mydataset(Dataset):
    def __init__(self):
        super().__init__()
        self.data = pd.read_csv('ChnSentiCorp_htl_all.csv')
        self.data = self.data.dropna()

    def __getitem__(self, index):
        return self.data.iloc[index,0], self.data.iloc[index,1]
    
    def __len__(self):
        return len(self.data)
    
dataset = Mydataset()

# print(dataset[0])
# print(len(dataset))

In [ ]:
from torch.utils.data import random_split

trainset, vaildset = random_split(dataset, lengths=[0.9, 0.1])
trainset, _ = random_split(trainset, lengths=[0.1, 0.9])
vaildset, _ = random_split(vaildset, lengths=[0.1, 0.9])

print(len(trainset))
print(len(vaildset))

tokenizer = AutoTokenizer.from_pretrained('uer/roberta-base-finetuned-dianping-chinese')

In [ ]:
import torch

def func(sample):
    label, text = [], []
    for item in sample:
        label.append(item[0])
        text.append(item[1])
    # print(label)
    # print(text)
    inputs = tokenizer(text, padding='longest', max_length=128, truncation=True, return_tensors='pt')
    inputs["labels"] = torch.tensor(label)

    return inputs

trainloader = DataLoader(trainset, batch_size=4, shuffle=True, collate_fn=func)
vaildloader = DataLoader(vaildset, batch_size=4, shuffle=False, collate_fn=func)
# print(len(trainloader))

# next(enumerate(trainloader))[1]

In [ ]:
from torch.optim import AdamW

model = AutoModelForSequenceClassification.from_pretrained('uer/roberta-base-finetuned-dianping-chinese')
model.cuda()
optimizer = AdamW(model.parameters(), lr=2e-5)


In [ ]:
from tqdm.auto import tqdm

def evalute(dataloader, model):
    model.eval()
    acc_num = 0
    with torch.inference_mode():
        for batch in dataloader:
            if torch.cuda.is_available:
                batch = {k:v.cuda() for k,v in batch.items()}
            outputs = model(**batch)
            pred = torch.argmax(outputs.logits, dim=-1)
            acc_num += (pred.long() == batch['labels'].long()).float().sum()
    return acc_num / len(vaildset)



def train(train_dataloader, vaild_dataloader, model, optimizer, epoch=3, log_step=50):
    global_step = 0
    for i in range(epoch):
        model.train()
        progress_bar = tqdm(range(len(train_dataloader)))
        progress_bar.set_description(f'loss: {0:>7f}')
        for batch in train_dataloader:
            if torch.cuda.is_available:
                batch = {k:v.cuda() for k,v in batch.items()}
            outputs = model(**batch)
            optimizer.zero_grad
            outputs.loss.backward()
            optimizer.step()
            if global_step % log_step == 0:
                print(f'global_step:{global_step} loss:{outputs.loss.item()}')
            global_step += 1
            progress_bar.set_description(f'loss: {0:>7f}')
            progress_bar.update(1)


        acc = evalute(vaild_dataloader, model)
        print(f"ep:{i+1} acc:{acc}")



In [ ]:
train(trainloader, vaildloader, model, optimizer)

In [ ]:
seq = "我觉得这家酒店不错，饭菜很好吃"

model.eval()

inputs = tokenizer(seq, padding='longest', truncation=True, max_length=128, return_tensors='pt')
inputs = {k:v.cuda() for k,v in inputs.items()}
outputs = model(**inputs).logits
pred = torch.argmax(outputs, dim=-1)
print(pred)

